In [1]:
from manipulation import ConfigureParser
from pydrake.all import (
    DiagramBuilder,
    Simulator,
    StartMeshcat,
    InverseKinematics,
    RotationMatrix,
    Solve,
    RigidTransform,
    Rgba,
    PiecewisePolynomial,
    TrajectorySource,
    KinematicTrajectoryOptimization,
    PositionConstraint,
    AddMultibodyPlantSceneGraph,
    MeshcatVisualizer,
    MeshcatVisualizerParams,
    Role,
    MinimumDistanceLowerBoundConstraint,
    BsplineTrajectory,
    Sphere,
    LeafSystem,
    AbstractValue,
    Context,
    BasicVector
)
from pydrake.perception import PointCloud
from manipulation.meshcat_utils import PublishPositionTrajectory
from manipulation.station import (
    LoadScenario,
    MakeHardwareStation,
    AddPointClouds,
)
from pathlib import Path
import numpy as np
from matplotlib import pyplot as plt
import trimesh
from controller import Controller, WsgController

from puzzle_pointclouds import (
    get_puzzle_and_tray_pointclouds,
    get_puzzle_pointcloud,
    get_tray_pointcloud,
    get_full_puzzle_pointcloud
)
import time
from puzzle_config import (
    camera_translation,
    cross_translation,
    infinity_translation,
    lower_left_translation,
    lower_right_translation,
    my_piece_translation,
    puzzle_center,
    puzzle_center_x,
    puzzle_center_y,
    puzzle_center_z,
    puzzle_offset,
    rectangle_translation,
    trapezoid_translation,
    tray_camera_translation,
    tray_translations,
    upper_left_translation,
    upper_right_translation,
    full_camera_translation,
    horizontal_camera_translation,
    vertical_camera_translation
)

from src.missing_piece_estimation import (
    find_closest_z_center,
    find_z_centers,
    largest_region,
    cloud_similarity,
)
from controller_visualization_functions import visualize_height_and_gradient, plot_point_cloud, crop_table


In [2]:
"""
Start meshcat and establish directory structure.
"""
meshcat = StartMeshcat()
def _format_vec(vec: tuple[float, float, float]) -> str:
    return f"[{vec[0]:.3f}, {vec[1]:.3f}, {vec[2]:.3f}]"
repo_root = Path("/Users/varun/robotics_final_project")
assets_dir = repo_root / "assets"


INFO:drake:Meshcat listening for connections at http://localhost:7001


In [ ]:
"Controller"

class Controller(LeafSystem):
    """PID controller for the IIWA robot"""

    def __init__(
        self, plant, iiwa, desired_z=0.12, resolution=0.005, step_gain=0.02
    ) -> None:
        LeafSystem.__init__(self)

        self.state_port = self.DeclareVectorInputPort("iiwa_state", 14)
        self.output_port = self.DeclareVectorOutputPort(
            "iiwa_torque", 7, self.ComputeTorque
        )
        self.hor_cloud_port = self.DeclareAbstractInputPort(
            "hor_cloud", AbstractValue.Make(PointCloud())
        )
        self.ver_cloud_port = self.DeclareAbstractInputPort(
            "ver_cloud", AbstractValue.Make(PointCloud())
        )

        self.plant = plant
        self.plant_context = plant.CreateDefaultContext()
        self.kp = 300
        self.kd = 200
        self.ki = 100
        self.qdot_desired = np.zeros(7)
        self.integral_error = np.zeros(7)
        self.iiwa = iiwa
        self.qs = None
        self.prev_time = 0.0
        self.idx = 0
        self.wsg_ctrl = None

        self.desired_z = desired_z
        self.step_gain = step_gain
        self.resolution = resolution

        self.Kp_pos, self.Kd_pos = 200, 20
        self.Kp_rot, self.Kd_rot = 200, 20

        self.wsg = plant.GetModelInstanceByName("wsg")
        self.gripper_body = plant.GetBodyByName("body", self.wsg)
        self.movement = False

    def _lookup_dir(self, x, y):
        i = int(np.clip(round((x - self.xs[0]) / self.resolution), 0, len(self.xs) - 1))
        j = int(np.clip(round((y - self.ys[0]) / self.resolution), 0, len(self.ys) - 1))
        return np.array([self.dir_x[j, i], self.dir_y[j, i]])

    def set_qs(self, qs):
        self.qs = qs

    def set_wsg_ctrl(self, wsg_ctrl):
        self.wsg_ctrl = wsg_ctrl

    def perception_processing(self, hor_data, ver_data):
        hor_points = hor_data.xyzs()
        ver_points = ver_data.xyzs()
        
        hor_points_filtered = hor_points[:, hor_points[1, :] <= -0.3]
        ver_points_filtered = ver_points[:, ver_points[0, :] <= 0.15]

        filtered_hor = PointCloud(new_size=hor_points_filtered.shape[1])
        filtered_hor.mutable_xyzs()[:] = hor_points_filtered

        filtered_ver = PointCloud(new_size=ver_points_filtered.shape[1])
        filtered_ver.mutable_xyzs()[:] = ver_points_filtered

        meshcat.SetObject(
            "hor_data",
            filtered_hor,
            point_size=0.01,
            rgba=Rgba(0.0, 1.0, 0.0),
        )

        meshcat.SetObject(
            "ver_data",
            filtered_ver,
            point_size=0.01,
            rgba=Rgba(0.0, 0.0, 1.0),
        )


    def ComputeTorque(self, context: Context, output: BasicVector) -> None:
        if self.qs is None:
            raise RuntimeError("initialize qs first")

        iiwa_state = self.state_port.Eval(context)

        # point cloud eval
        

        q = iiwa_state[:7]
        qdot = iiwa_state[7:]

        self.plant.SetPositions(self.plant_context, self.iiwa, q)
        self.plant.SetVelocities(self.plant_context, self.iiwa, qdot)

        q_des = self.qs[self.idx]
        err_norm = np.linalg.norm(q_des - q)

        if err_norm < 0.01 and self.idx < len(self.qs) - 1:
            if self.idx == 1:
                self.wsg_ctrl.set_target(0)
            self.idx += 1
            q_des = self.qs[self.idx]
        elif err_norm < 0.01 and self.idx == len(self.qs) - 1:
            if not self.movement:
                print("Done! Holding final position")
                self.movement = True
                self.final_q = q.copy()  # Lock in the final position
            print("Calibrating")
            hor_data = self.hor_cloud_port.Eval(context)
            ver_data = self.ver_cloud_port.Eval(context)
            self.perception_processing(hor_data,ver_data)
            
            q_des = self.final_q  # Use the locked position, not current q

        current_time = context.get_time()
        dt = current_time - self.prev_time

        position_error = q_des - q
        velocity_error = self.qdot_desired - qdot

        if dt > 0:
            self.integral_error += dt * position_error

        torque = (
            self.kp * position_error
            + self.kd * velocity_error
            + self.ki * self.integral_error
        )
        tau_g_full = self.plant.CalcGravityGeneralizedForces(self.plant_context)
        self.prev_time = current_time

        output.set_value(torque - tau_g_full[:7])


class WsgController(LeafSystem):
    def __init__(self, target_width=0.06, kp=400.0, kd=50.0):
        super().__init__()
        self.target = target_width / 2.0  # each finger
        self.kp = kp
        self.kd = kd

        self.state_port = self.DeclareVectorInputPort("wsg_state", 4)
        self.DeclareVectorOutputPort("wsg_actuation", 2, self.CalcTau)

    def set_target(self, width):
        self.target = width / 2.0

    def CalcTau(self, context, output):
        if self.target == 0:
            output.SetFromVector([3.4575, -3.4575])  # closes the gripper via forces

        else:
            x = self.state_port.Eval(context).ravel()
            q_l, q_r, v_l, v_r = x  # get inputs from input port
            qd = self.target  #
            tau_l = (
                self.kp * (-qd - q_l) - self.kd * v_l
            )  # pid control formula to find l and r forces
            tau_r = self.kp * (qd - q_r) - self.kd * v_r
            output.SetFromVector([tau_l, tau_r])


In [4]:
# assets for tray pieces
my_piece_sdf_uri = (assets_dir / "my_piece.sdf").resolve().as_uri()
rectangle_sdf_uri = (assets_dir / "rectangle.sdf").resolve().as_uri()
trapezoid_sdf_uri = (assets_dir / "trapezoid.sdf").resolve().as_uri()
infinity_sdf_uri = (assets_dir / "infinity.sdf").resolve().as_uri()

# assets for welded puzzle frame
corner_sdf_uri = (assets_dir / "puzzle_corner.sdf").resolve().as_uri()
cross_sdf_uri = (assets_dir / "puzzle_cross.sdf").resolve().as_uri()


scenario_string = f"""directives:
- add_model:
    name: iiwa
    file: package://drake_models/iiwa_description/urdf/iiwa14_primitive_collision.urdf
    default_joint_positions:
      iiwa_joint_1: [-1.57]
      iiwa_joint_2: [0.1]
      iiwa_joint_3: [0]
      iiwa_joint_4: [-1.2]
      iiwa_joint_5: [0]
      iiwa_joint_6: [1.6]
      iiwa_joint_7: [0]
- add_weld:
    parent: world
    child: iiwa::iiwa_link_0

- add_model:
    name: wsg
    file: package://manipulation/hydro/schunk_wsg_50_with_tip.sdf
- add_weld:
    parent: iiwa::iiwa_link_7
    child: wsg::body
    X_PC:
        translation: [0, 0, 0.09]
        rotation: !Rpy {{ deg: [90, 0, 90]}}

- add_model:
    name: table
    file: "{(repo_root / 'table.sdf').resolve().as_uri()}"
- add_weld:
    parent: world
    child: table::table_link
    X_PC:
        translation: [0.0, 0.0, -0.05]
        rotation: !Rpy {{ deg: [0, 0, -90] }}


- add_model:
    name: custom_rectangle
    file: "{rectangle_sdf_uri}"
- add_weld:
    parent: world
    child: custom_rectangle::my_piece_link
    X_PC:
        translation: {_format_vec(rectangle_translation)}
        rotation: !Rpy {{ deg: [0, 0, 0] }}
- add_model:
    name: custom_my_piece
    file: "{my_piece_sdf_uri}"
- add_weld:
    parent: world
    child: custom_my_piece::my_piece_link
    X_PC:
        translation: {_format_vec(my_piece_translation)}
        rotation: !Rpy {{ deg: [0, 0, 0] }}

- add_model:
    name: trapezoid
    file: "{trapezoid_sdf_uri}"
- add_weld:
    parent: world
    child: trapezoid::trapezoid_link
    X_PC:
        translation: {_format_vec(trapezoid_translation)}
        rotation: !Rpy {{ deg: [0, 0, 0] }}

- add_model:
    name: infinity
    file: "{infinity_sdf_uri}"
- add_weld:
    parent: world
    child: infinity::infinity_link
    X_PC:
        translation: {_format_vec(infinity_translation)}
        rotation: !Rpy {{ deg: [0, 0, 0] }}

- add_model:
    name: puzzle_upper_right
    file: "{corner_sdf_uri}"
- add_weld:
    parent: world
    child: puzzle_upper_right::corner_link
    X_PC:
        translation: {_format_vec(upper_right_translation)}
        rotation: !Rpy {{ deg: [0, 0, 0] }}
- add_model:
    name: puzzle_upper_left
    file: "{corner_sdf_uri}"
- add_weld:
    parent: world
    child: puzzle_upper_left::corner_link
    X_PC:
        translation: {_format_vec(upper_left_translation)}
        rotation: !Rpy {{ deg: [0, 0, 90] }}
- add_model:
    name: puzzle_lower_left
    file: "{corner_sdf_uri}"
- add_weld:
    parent: world
    child: puzzle_lower_left::corner_link
    X_PC:
        translation: {_format_vec(lower_left_translation)}
        rotation: !Rpy {{ deg: [0, 0, 180] }}
- add_model:
    name: puzzle_lower_right
    file: "{corner_sdf_uri}"
- add_weld:
    parent: world
    child: puzzle_lower_right::corner_link
    X_PC:
        translation: {_format_vec(lower_right_translation)}
        rotation: !Rpy {{ deg: [0, 0, -90] }}
- add_model:
    name: puzzle_cross
    file: "{cross_sdf_uri}"
    default_free_body_pose:
        cross_link:
            translation: {_format_vec(cross_translation)}
            rotation: !Rpy {{ deg: [0, 0, 0] }}

- add_model:
    name: puzzle_camera
    file: "package://manipulation/camera_box.sdf"
- add_weld:
    parent: world
    child: puzzle_camera::base
    X_PC:
        translation: {_format_vec(camera_translation)}
        rotation: !Rpy {{ deg: [-160, 0, 0] }}

- add_model:
    name: tray_camera
    file: "package://manipulation/camera_box.sdf"
- add_weld:
    parent: world
    child: tray_camera::base
    X_PC:
        translation: {_format_vec(tray_camera_translation)}
        rotation: !Rpy {{ deg: [-150, 0, 0] }}

- add_model:
    name: horizontal_calibration
    file: "package://manipulation/camera_box.sdf"
- add_weld:
    parent: world
    child: horizontal_calibration::base
    X_PC:
        translation: {_format_vec(horizontal_camera_translation)}
        rotation: !Rpy {{ deg: [-90, 0, 0] }}

- add_model:
    name: vertical_calibration
    file: "package://manipulation/camera_box.sdf"
- add_weld:
    parent: world
    child: vertical_calibration::base
    X_PC:
        translation: {_format_vec(vertical_camera_translation)}
        rotation: !Rpy {{ deg: [-90, 0, -90] }}

cameras:
  puzzle_camera:
    name: camera_puzzle
    depth: true
    X_PB:
        base_frame: puzzle_camera::base

  tray_camera:
    name: camera_tray
    depth: true
    X_PB:
        base_frame: tray_camera::base

  horizontal_calibration:
    name: horizontal_calibration
    depth: true
    X_PB:
        base_frame: horizontal_calibration::base

  vertical_calibration:
    name: vertical_calibration
    depth: true
    X_PB:
        base_frame: vertical_calibration::base
"""

"""
Fully create simulation environment and build core diagram.
"""

meshcat.Delete()
scenario = LoadScenario(data=scenario_string)
station = MakeHardwareStation(scenario)
builder = DiagramBuilder()

station_sys = builder.AddSystem(station)

# add point clouds
pcd_systems = AddPointClouds(builder=builder, station=station_sys, scenario=scenario)
puzzle_pcd_sys = pcd_systems["camera_puzzle"]
tray_pcd_sys = pcd_systems["camera_tray"]
hor_sys = pcd_systems["horizontal_calibration"]
ver_sys = pcd_systems["vertical_calibration"]

puzzle_pcd_port = puzzle_pcd_sys.point_cloud_output_port()
tray_pcd_port = tray_pcd_sys.point_cloud_output_port()
hor_port = hor_sys.point_cloud_output_port()
ver_port = ver_sys.point_cloud_output_port()

builder.ExportOutput(puzzle_pcd_port, "puzzle.point_cloud")
builder.ExportOutput(tray_pcd_port, "tray.point_cloud")
builder.ExportOutput(hor_port, "hor.point_cloud")
builder.ExportOutput(ver_port, "ver.point_cloud")


scene_graph = station_sys.GetSubsystemByName("scene_graph")
plant = station_sys.GetSubsystemByName("plant")

visualizer = MeshcatVisualizer.AddToBuilder(
    builder,
    station.GetOutputPort("query_object"),
    meshcat,
    MeshcatVisualizerParams(role=Role.kIllustration),
)
collision_visualizer = MeshcatVisualizer.AddToBuilder(
    builder,
    station.GetOutputPort("query_object"),
    meshcat,
    MeshcatVisualizerParams(
        prefix="collision", role=Role.kProximity, visible_by_default=False
    ),
)

wsg = plant.GetModelInstanceByName("wsg")
iiwa = plant.GetModelInstanceByName("iiwa")
gripper_frame = plant.GetFrameByName("body")

wsg_ctrl = builder.AddSystem(WsgController(target_width=0.06))
builder.Connect(station.GetOutputPort("wsg_state"), wsg_ctrl.state_port)
builder.Connect(wsg_ctrl.get_output_port(0), station.GetInputPort("wsg_actuation"))

iiwa_ctrl = builder.AddSystem(Controller(plant, iiwa))
builder.Connect(station.GetOutputPort("iiwa_state"), iiwa_ctrl.state_port)
builder.Connect(iiwa_ctrl.get_output_port(0), station.GetInputPort("iiwa_actuation"))
builder.Connect(hor_port, iiwa_ctrl.hor_cloud_port)
builder.Connect(ver_port, iiwa_ctrl.ver_cloud_port)

diagram = builder.Build()
context = diagram.CreateDefaultContext()
plant_context = plant.CreateDefaultContext()
diagram.ForcedPublish(context) 


In [5]:
from puzzle_pointclouds import get_hor_pointcloud, get_ver_pointcloud


full_puzzle_cloud = get_puzzle_pointcloud(diagram, context)
full_tray_cloud = get_tray_pointcloud(diagram, context)
hor_cloud = get_hor_pointcloud(diagram, context)
ver_cloud = get_ver_pointcloud(diagram, context)

puzzle_cloud, tray_clouds = get_puzzle_and_tray_pointclouds(
    diagram,
    context,
    puzzle_center=puzzle_center,
    tray_translations=tray_translations,
)




In [6]:
puzzle_points = puzzle_cloud.xyzs().T

tray_piece_tight_clouds = {}  # dict to map name of piece to refined positive clouds
for piece in tray_clouds:
    cloud = tray_clouds[piece]
    points = cloud.xyzs().T
    center1, center2 = find_z_centers(puzzle_points)

    min_center = min(center1, center2)
    max_center = max(center1, center2)

    # we want max center now
    positive_space_points = []
    for point in points:
        closest_center = find_closest_z_center(point, min_center, max_center)
        if closest_center == max_center:
            positive_space_points.append(point)

    pos = largest_region(positive_space_points)

    cloud_pos = PointCloud(new_size=pos.shape[0])
    cloud_pos.mutable_xyzs()[:] = pos.T

    tray_piece_tight_clouds[piece] = pos


# Identify negative space
center1, center2 = find_z_centers(puzzle_points)
min_center = min(center1, center2)  # corresponds to negative space
max_center = max(center1, center2)  # corresponds to boundary puzzle pieces

negative_space_points = []
for point in puzzle_points:
    closest_center = find_closest_z_center(point, min_center, max_center)
    if closest_center == min_center:
        negative_space_points.append(point)

# now choose largest continuous region for these negative space points

neg_pts = largest_region(negative_space_points)

cloud_neg = PointCloud(new_size=neg_pts.shape[0])

cloud_neg_avg = neg_pts.mean(axis=0)
cloud_neg.mutable_xyzs()[:] = neg_pts.T


In [7]:
scores = {}
# Compute similarity scores between tray pieces and missing piece
for piece, pos_pts in tray_piece_tight_clouds.items():
    print(f"######## {piece} cloud and negative space cloud similarity score ########")
    score, newB, R, t = cloud_similarity(neg_pts, pos_pts)
    print(f"Score: {score}")
    scores[piece] = {"score": score, "rotation": R, "translation": t, "cloud": pos_pts}
    if piece == "cross":
        cloud_translated = PointCloud(new_size=newB.shape[0])
        cloud_translated.mutable_xyzs()[:] = newB.T
        
        print(f"Rotation Matrix: {R}")
        print(f"Translation: {t}")
best_piece, best_entry = max(scores.items(), key=lambda item: item[1]["score"])
cloud = best_entry["cloud"]
piece_location = cloud.mean(axis=0)



######## rectangle cloud and negative space cloud similarity score ########
Score: 49.41881788196033
######## my_piece cloud and negative space cloud similarity score ########
Score: 49.41881788196033
######## trapezoid cloud and negative space cloud similarity score ########
Score: 1049.5884791290944
######## infinity cloud and negative space cloud similarity score ########
Score: 906.5201691640135
######## cross cloud and negative space cloud similarity score ########
Score: 684.6339059953385
Rotation Matrix: [[ 9.99999829e-01  5.85480089e-04]
 [-5.85480089e-04  9.99999829e-01]]
Translation: [-0.39704009  0.00231723]


In [8]:
import numpy as np
translation = np.append(t, 0.00)
print(translation)

[-0.39704009  0.00231723  0.        ]


In [9]:

def solve_ik(X_WG_target, orientation_tolerance=0.001, pos_tol=0.001):
    """
    Solve IK for a target pose.
    
    Args:
        X_WG_target: Target RigidTransform for gripper in world frame
        context: Plant context
        orientation_tolerance: Tolerance for orientation in radians
        pos_tol: Tolerance for position in meters (default 1cm)
    """
    ik = InverseKinematics(plant, plant_context)
    q = ik.q()

    # Position constraint: point on gripper at target position
    p_W = X_WG_target.translation()
    ik.AddPositionConstraint(
        gripper_frame, np.array([0, 0.1, 0]),
        plant.world_frame(),
        p_W - pos_tol, p_W + pos_tol
    )

    R_WG_des = RotationMatrix.MakeXRotation(-np.pi / 2)  # flip around X so z points down
    ik.AddOrientationConstraint(
        gripper_frame, RotationMatrix(),
        plant.world_frame(), R_WG_des,
        orientation_tolerance
    )

    # Use current configuration as initial guess
    q_current = plant.GetPositions(plant_context)
    ik.prog().SetInitialGuess(q, q_current)
    
    result = Solve(ik.prog())
    if not result.is_success():
        print(f"IK failed for target pose {p_W}")
        print(f"Solver: {result.get_solver_id().name()}")
        print(f"Current config: {q_current}")
        raise RuntimeError("IK failed for target pose")
    return result.GetSolution(q)


In [10]:
q_start = np.array([-1.57, 0.1, 0, -1.2, 0, 1.6, 0])

# descend
X_WGoal = RigidTransform(cross_translation + np.array([0, 0.033, 0.02]))
q_descend = solve_ik(X_WGoal)[:7]

# lift
lift = cross_translation + np.array([0, 0.033, 0.08])
X_WGoal = RigidTransform(lift) # just lift it slightly above
q_lift = solve_ik(X_WGoal)[:7]

# translate it by results from perception pipeline
# initial move to puzzle before nudging
initial_puzzle_pred = lift + translation
X_WGoal = RigidTransform(initial_puzzle_pred)
q_goal = solve_ik(X_WGoal)[:7]

In [ ]:
iiwa_ctrl.set_wsg_ctrl(wsg_ctrl)
iiwa_ctrl.set_qs([q_start, q_descend, q_lift, q_goal])

simulator = Simulator(diagram)
simulator.set_target_realtime_rate(0.5)
simulator.AdvanceTo(70)

Done! Holding final position
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating
Calibrating